# 018 — Hallucination Detection
**Evaluation + Observability** | Detect unsupported claims before surfacing answers

Two complementary approaches:
1. **NLI-based** (Natural Language Inference) — cross-encoder checks if context *entails* the answer
2. **LLM self-critique** — ask the LLM to verify each claim in its own answer
3. **Sentence-level scoring** — flag individual sentences, not just the full answer


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build RAG pipeline to generate test answers ──────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c)
        for text in [ai_text, ml_text, dl_text]
        for c in splitter.split_text(text[:12000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})

def rag_with_context(question: str):
    retrieved = retriever.invoke(question)
    ctx       = "\n\n".join(d.page_content for d in retrieved)
    answer    = (
        ChatPromptTemplate.from_template(
            "Answer using only context.\nContext:\n{ctx}\nQ: {q}\nA:"
        ) | llm | StrOutputParser()
    ).invoke({"ctx": ctx[:3000], "q": question})
    return answer, ctx

print(f"✓ RAG pipeline over {len(docs)} docs")


In [ ]:
# ── Method 1: NLI cross-encoder hallucination score ──────────────────────
from sentence_transformers import CrossEncoder

# NLI model: given (premise, hypothesis) → label (entailment/neutral/contradiction)
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")
print("✓ NLI model loaded: cross-encoder/nli-deberta-v3-small")

def nli_hallucination_score(context: str, answer: str) -> dict:
    '''
    Returns entailment probability.
    entailment → answer is supported  (low hallucination)
    contradiction → answer contradicts context (high hallucination)
    neutral → answer is not directly supported
    '''
    # Score: [contradiction, neutral, entailment]
    scores = nli_model.predict([(context[:1000], answer[:500])])
    probs  = scores[0]
    labels = ["contradiction", "neutral", "entailment"]
    result = dict(zip(labels, probs.tolist()))
    result["hallucination_risk"] = 1 - result["entailment"]
    return result

# Test: grounded vs hallucinated answer
q = "What is supervised learning?"
real_answer, context = rag_with_context(q)
fake_answer = "Supervised learning was invented in 1943 by Alan Turing in Cambridge."

print("Grounded answer NLI score:")
print(nli_hallucination_score(context, real_answer))
print("\nHallucinated answer NLI score:")
print(nli_hallucination_score(context, fake_answer))


In [ ]:
# ── Method 2: LLM self-critique per sentence ─────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

sentence_check_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a fact-checker. Given a context and a claim, decide if the claim is:\n"
        "- 'supported': directly supported by the context\n"
        "- 'unsupported': not in the context (but not necessarily wrong)\n"
        "- 'contradicted': the context says something different\n"
        "Reply with only the label."
    )),
    ("human", "Context:\n{context}\n\nClaim: {claim}"),
])
sentence_checker = sentence_check_prompt | llm | StrOutputParser()

def check_answer_sentence_by_sentence(answer: str, context: str):
    sentences = [s.strip() for s in answer.split(".") if len(s.strip()) > 20]
    results   = []
    for sent in sentences:
        verdict = sentence_checker.invoke({"context": context[:2000], "claim": sent})
        results.append({"sentence": sent, "verdict": verdict.strip().lower()})
    return results

answer, context = rag_with_context("How does gradient descent optimize neural networks?")
print(f"Answer: {answer[:300]}\n")
print("Sentence-level check:")
checks = check_answer_sentence_by_sentence(answer, context)
for c in checks:
    icon = {"supported": "✓", "unsupported": "?", "contradicted": "✗"}.get(c["verdict"], "·")
    print(f"  {icon} [{c['verdict']:12}] {c['sentence'][:100]}")


In [ ]:
# ── Method 3: Pipeline — detect before surfacing ──────────────────────────
def safe_rag_answer(question: str, hallucination_threshold=0.4):
    answer, context = rag_with_context(question)

    # Quick NLI check
    nli_result = nli_hallucination_score(context, answer)
    risk = nli_result["hallucination_risk"]

    if risk > hallucination_threshold:
        # Regenerate with stricter prompt
        strict_chain = (
            ChatPromptTemplate.from_template(
                "Answer ONLY using exact quotes from the context. "
                "Do NOT add any information not in the context.\n\n"
                "Context:\n{ctx}\n\nQ: {q}\n\nA:"
            ) | llm_smart | StrOutputParser()
        )
        answer = strict_chain.invoke({"ctx": context[:3000], "q": question})
        nli_result2 = nli_hallucination_score(context, answer)
        return {"answer": answer, "risk_before": risk, "risk_after": nli_result2["hallucination_risk"], "regenerated": True}

    return {"answer": answer, "risk_before": risk, "risk_after": risk, "regenerated": False}

questions = [
    "What is the history of machine learning?",
    "How does BERT use masked language modeling?",
]
for q in questions:
    r = safe_rag_answer(q)
    print(f"Q: {q}")
    print(f"  Hallucination risk: {r['risk_before']:.3f} → {r['risk_after']:.3f}")
    print(f"  Regenerated: {r['regenerated']}")
    print(f"  Answer: {r['answer'][:200]}\n")


## Key Takeaways
| Method | Latency | Accuracy | Use case |
|---|---|---|---|
| NLI cross-encoder | ~50ms | High | Production gating |
| LLM self-critique | ~2s | Very high | High-stakes answers |
| Sentence-level NLI | ~200ms | High | Detailed audit trail |

- Gate on `hallucination_risk > 0.4`: regenerate with stricter prompt
- Log all rejected/regenerated answers — they reveal retrieval gaps
- Pair with **RAGAS faithfulness** (notebook 017) for systematic evaluation
